In [1]:
from topic_gen.generate import Generator
import ir_datasets
import pandas as pd
import os
from topic_gen.models import TRECTopic, Topics
from langchain.chat_models import init_chat_model

from topic_gen import logger
logger.setLevel("INFO")

In [2]:
class uqv_parser:
    def __init__(self):
        self.queries = []
        self.dataset = ir_datasets.load("disks45/nocr/trec-robust-2004")
        self.store = self.dataset.docs_store()

        self.qrels_map = self.make_qrels_map()

    def make_qrels_map(self):
        qrels_map = {}
        for qrel in self.dataset.qrels_iter():
            if not qrels_map.get(qrel.query_id):
                qrels_map[qrel.query_id] = []
            if qrel.relevance > 0:
                doc = self.store.get(qrel.doc_id)
                qrels_map[qrel.query_id].append(doc.body)
        return qrels_map

    def parse_variants(self):
        # variants
        uqv = pd.read_csv(
            "../data/raw/trec-reference/robust-uqv.txt", sep=";", names=["query_id", "uqv"]
        )
        uqv["qid"] = uqv["query_id"].apply(lambda x: x.split("-")[0])

        for query in self.dataset.queries_iter():
            variants = uqv[uqv["qid"] == query.query_id]["uqv"].to_list()

            self.queries.append(
                {
                    "qid": query.query_id,
                    "title": query.title,
                    "description": query.description,
                    "narrative": query.narrative,
                    "uqv": variants,
                    "rel_docs": self.qrels_map.get(query.query_id),
                }
            )

        return self.queries

In [3]:
parser = uqv_parser()
topics = parser.parse_variants()

In [4]:
!export VLLM_TARGET_DEVICE=cpu
!export VLLM_BUILD_WITH_CUDA=0

In [5]:
# from vllm.entrypoints.llm import LLM, SamplingParams
import torch

# import sys
# sys.path.append("../vllm")

from langchain_community.llms import VLLM


use_mps = torch.backends.mps.is_available()
device_type = "mps" if use_mps else "cpu"
print(f"Using device: {device_type}")

model_dir = "../models"
model_name = "Qwen/Qwen3-0.6B"
llm = VLLM(model=model_name, 
          download_dir=model_dir,
          trust_remote_code=True,
          vllm_kwargs={
            "max_num_batched_tokens": 20960, 
            "max_model_len":20960
            },
          # max_num_batched_tokens=20000,
        #   tensor_parallel_size=1,
        #   trust_remote_code=True,
        #   dtype="float16" if use_mps else "float32")
)

Using device: mps


/Users/jueri/dev/conf26-generating-topics/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 07-27 16:44:08 [__init__.py:235] Automatically detected platform cpu.
WARNING 07-27 16:44:09 [_custom_ops.py:20] Failed to import from vllm._C with ImportError("dlopen(/Users/jueri/dev/conf26-generating-topics/.venv/lib/python3.10/site-packages/vllm/_C.abi3.so, 0x0002): symbol not found in flat namespace '__Z14int8_scaled_mmRN2at6TensorERKS0_S3_S3_S3_RKNSt3__18optionalIS0_EE'")
WARNING 07-27 16:44:15 [config.py:3392] Your device 'cpu' doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 07-27 16:44:15 [config.py:3443] Casting torch.bfloat16 to torch.float16.
INFO 07-27 16:44:15 [config.py:1604] Using max model len 20960
WARNING 07-27 16:44:15 [cpu.py:113] Environment variable VLLM_CPU_KVCACHE_SPACE (GiB) for CPU backend is not set, using 4 by default.
INFO 07-27 16:44:15 [arg_utils.py:1030] Chunked prefill is not supported for ARM and POWER CPUs; disabling it for V1 backend.


2025-07-27 16:44:16,728	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 07-27 16:44:20 [__init__.py:235] Automatically detected platform cpu.
WARNING 07-27 16:44:21 [_custom_ops.py:20] Failed to import from vllm._C with ImportError("dlopen(/Users/jueri/dev/conf26-generating-topics/.venv/lib/python3.10/site-packages/vllm/_C.abi3.so, 0x0002): symbol not found in flat namespace '__Z14int8_scaled_mmRN2at6TensorERKS0_S3_S3_S3_RKNSt3__18optionalIS0_EE'")
INFO 07-27 16:44:21 [core.py:572] Waiting for init message from front-end.
INFO 07-27 16:44:21 [core.py:71] Initializing a V1 LLM engine (v0.10.0) with config: model='Qwen/Qwen3-0.6B', speculative_config=None, tokenizer='Qwen/Qwen3-0.6B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=20960, download_dir='../models', load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=True, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_c

Process EngineCore_0:
Traceback (most recent call last):
  File "/Users/jueri/.pyenv/versions/3.10.0/lib/python3.10/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/Users/jueri/.pyenv/versions/3.10.0/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/jueri/dev/conf26-generating-topics/.venv/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 636, in run_engine_core
    raise e
  File "/Users/jueri/dev/conf26-generating-topics/.venv/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 623, in run_engine_core
    engine_core = EngineCoreProc(*args, **kwargs)
  File "/Users/jueri/dev/conf26-generating-topics/.venv/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 441, in __init__
    super().__init__(vllm_config, executor_class, log_stats,
  File "/Users/jueri/dev/conf26-generating-topics/.venv/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 77, in __init__
 

RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {'EngineCore_0': 1}

In [5]:
# Generator
# llm = init_chat_model(
#     model = "gemini-2.5-flash",
#     model_provider="google_genai",
#     temperature=0,
# )

# Ollama
# 36, 33 sec on one GPU
# llm = init_chat_model(
#         model="llama3.3:70b",
#         temperature=0.0,
#         base_url="http://139.6.160.39:11434/v1",
#         api_key="not-needed",
#         model_provider="openai",
#     )


# vLLM
# 38, 36, 32 sec on one GPU
# 21 sec on 2 GPUs
# llm = init_chat_model(
#         model="lambdalabs/Llama-3.3-70b-Instruct-AWQ-4bit",
#         temperature=0.0,
#         base_url="http://139.6.160.39:6543/v1",
#         api_key="not-needed",
#         model_provider="openai",
#     )
generator = Generator(llm=llm, output_class=TRECTopic)

n_queries = 3
n_docs = 1

for topic in topics:
    generated_topic = generator.generate(
        prompt_name = "trec-base",
        number_of_topics=1,
        queries="\n".join(topic["uqv"][:n_queries]),
        relevant_documents="\n\n".join(topic["rel_docs"][:n_docs]),
    )
    break

[topic_gen] [INFO] Final text prompt:
 Objective: Generate a high-quality, structured TREC-style topic that simulates a nuanced user information need. This topic will be synthesized from a given set of user search queries and a collection of text snippets from documents the user has identified as relevant.
Instructions:
You are an expert in Information Science and a seasoned analyst for the Text REtrieval Conference (TREC). Your task is to create a formal TREC topic that represents the underlying information need of a user. You will be provided with two pieces of evidence: a list of queries the user has searched for, and a set of excerpts from documents they have judged as relevant.
Your goal is to move beyond the specific keywords in the queries and the explicit text in the documents to define the users broader, more abstract information goal. You must articulate what makes a document truly relevant to this users need, including the specific criteria for inclusion and exclusion.

Cont

In [21]:
generated_topic

TRECTopic(title='International Crime Organizations', description='Find information on the activities, structures, and international law enforcement efforts related to organized crime groups.', narrative='A relevant document must discuss specific international criminal organizations, their known activities (such as drug trafficking), and interactions with legal authorities across national borders. To be considered relevant, a document needs to provide details on either the organizational structure of these groups, their illegal activities, or the challenges faced by law enforcement agencies in prosecuting them. Documents that merely mention crime or international relations without focusing on organized criminal activity would not be relevant.')

In [15]:
generated_topic

TRECTopic(title='International Crime Organizations', description='Find information on the activities, legal challenges, and international cooperation related to organized crime groups, particularly those involved in drug trafficking.', narrative='A relevant document must discuss specific international criminal organizations, their activities (such as drug trafficking), and the legal or political challenges related to their prosecution. This includes but is not limited to, details on international cooperation, evidence collection, extradition processes, and any notable cases or individuals involved. To be relevant, a document needs to provide more than general information on organized crime; it must delve into the specifics of how these organizations operate internationally and the efforts to combat them.')

In [5]:
print("Title:")
print(topic["title"], topic["uqv"][:3])
print(generated_topic.title)

print("\nDescription:")
print(topic["description"])
print(generated_topic.description)

print("\nNarrative:")
print(topic["narrative"])
print(generated_topic.narrative)


Title:
International Organized Crime ['organizations international criminal activity', 'international criminal organizations', 'international organized crime']
International Criminal Organizations

Description:
Identify organizations that participate in international criminal
activity, the activity, and, if possible, collaborating organizations
and the countries involved.
Find information identifying specific international criminal organizations and the types of illegal activities they engage in.

Narrative:
A relevant document must as a minimum identify the organization and the
type of illegal activity (e.g., Columbian cartel exporting cocaine).
Vague references to international drug trade without identification of
the organization(s) involved would not be relevant.
A relevant document must identify a specific international criminal organization and the type of illegal activity it is involved in. For example, a document discussing the Cali cartel and its drug trafficking operations wo

In [6]:
generated_topic

TRECTopic(title='International Criminal Organizations', description='Find information identifying specific international criminal organizations and the types of illegal activities they engage in.', narrative='A relevant document must identify a specific international criminal organization and the type of illegal activity it is involved in. For example, a document discussing the Cali cartel and its drug trafficking operations would be highly relevant. Documents that only discuss general crime, domestic criminal activity without international connections, or do not name a specific organization are not relevant. Information about the leaders, structure, or legal challenges related to these organizations is also relevant.')

In [14]:
topic

{'qid': '301',
 'title': 'International Organized Crime',
 'description': 'Identify organizations that participate in international criminal\nactivity, the activity, and, if possible, collaborating organizations\nand the countries involved.',
 'narrative': 'A relevant document must as a minimum identify the organization and the\ntype of illegal activity (e.g., Columbian cartel exporting cocaine).\nVague references to international drug trade without identification of\nthe organization(s) involved would not be relevant.',
 'uqv': ['organizations international criminal activity',
  'international criminal organizations',
  'international organized crime',
  'international crime groups',
  'international drug trades crime group in south america',
  'international smuggling group',
  'international criminal organizations',
  'international illegal drug trade organizations country',
  'international terrorist groups illegal activity country',
  'international organizations benefiting from c

In [ ]:
def generate_topics_robust_uqv(
    topics,
    model = "gemini-2.5-flash",
    prompt_name = "trec-base",
    dataset_name = "robust04",
    n_queries = 3,
    n_docs = 1,
    output=".",
):
    # Generator
    llm = init_chat_model(
        model=model,
        model_provider="google_genai",
        temperature=0,
    )
    generator = Generator(llm=llm, output_class=TRECTopic)


    for topic in topics:
        generated_topic = generator.generate(
            prompt_name=prompt_name,
            number_of_topics=1,
            queries="\n".join(topic["uqv"][:n_queries]),
            clicked_documents="\n\n".join(topic["rel_docs"][:n_docs]),
        )
        
        topic_dict = generated_topic.model_dump()
        topic_dict["qid"] = topic["qid"]
        
        # Save
        with open(os.path.join(output, f"topics-{dataset_name}-{model}-{prompt_name}-q{n_queries}-d{n_docs}.jsonl"), "a+") as f:
            f.write(f"{topic_dict}\n")

In [ ]:
parser = uqv_parser()
topics = parser.parse_variants()

In [6]:
generate_topics_robust_uqv(
    topics=topics[200:210], 
    output="../experiments/data/interim/",
    n_queries=1,
    n_docs=1,
    )

[topic_gen] [INFO] Final text prompt:
 Objective:
To generate a high-quality, structured TREC-style topic that simulates a nuanced user information need. This topic will be synthesized from a given set of user search queries and a collection of text snippets from documents the user has identified as relevant.
Instructions:
You are an expert in Information Science and a seasoned analyst for the Text REtrieval Conference (TREC). Your task is to create a formal TREC topic that represents the underlying information need of a user. You will be provided with two pieces of evidence: a list of queries the user has searched for, and a set of excerpts from documents they have judged as relevant.
Your goal is to move beyond the specific keywords in the queries and the explicit text in the documents to define the users broader, more abstract information goal. You must articulate what makes a document truly relevant to this users need, including the specific criteria for inclusion and exclusion.

C

In [8]:
# generate_topics_robust_uqv(
#     topics=topics[200:210], 
#     output="../experiments/data/interim/",
#     n_queries=2,
#     n_docs=1,
#     )

# generate_topics_robust_uqv(
#     topics=topics[200:210], 
#     output="../experiments/data/interim/",
#     n_queries=3,
#     n_docs=1,
#     )
generate_topics_robust_uqv(
    topics=topics[200:210], 
    output="../experiments/data/interim/",
    n_queries=4,
    n_docs=1,
    )
generate_topics_robust_uqv(
    topics=topics[200:210], 
    output="../experiments/data/interim/",
    n_queries=5,
    n_docs=1,
    )

[topic_gen] [INFO] Final text prompt:
 Objective:
To generate a high-quality, structured TREC-style topic that simulates a nuanced user information need. This topic will be synthesized from a given set of user search queries and a collection of text snippets from documents the user has identified as relevant.
Instructions:
You are an expert in Information Science and a seasoned analyst for the Text REtrieval Conference (TREC). Your task is to create a formal TREC topic that represents the underlying information need of a user. You will be provided with two pieces of evidence: a list of queries the user has searched for, and a set of excerpts from documents they have judged as relevant.
Your goal is to move beyond the specific keywords in the queries and the explicit text in the documents to define the users broader, more abstract information goal. You must articulate what makes a document truly relevant to this users need, including the specific criteria for inclusion and exclusion.

C

AttributeError: 'NoneType' object has no attribute 'model_dump'